# Aromas piloto 2026 — producción calibrada sólo con vino

## tl;dr

Los parámetros biológicos se estiman exclusivamente desde la química líquida. El condensado no participa en el objetivo y se utiliza como prueba independiente del contrato `eta_AB=1` más partición de Morakul/Mouret.

## Contexto y métodos

### Supuestos clave

- Captura conjunta A+B fijada a 1 por diseño, no estimada.
- Producción calibrada únicamente contra vino.
- Condensado completamente excluido del optimizador.
- Transferencia fijada como `K(E,T)·QCO2`.
- Validación leave-one-reactor-run-out en los seis procesos.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'fermentation_model').exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / 'fermentation_model'))
from pilot_2026 import run_aroma_wine_calibrated_complete_capture_2026 as analysis

result = analysis.load_results()
print('Veredicto:', result['gate']['verdict'])
print('Modelo diagnóstico:', result['gate']['diagnostic_best_model'])

## Resultados

In [ ]:
primary = result['metrics'].query("analysis_policy == 'primary'")
display(primary.round(4))
display(result['comparison'].round(4))
display(Image(filename=analysis.FIGURE_DIR / '02_loro_nrmse_comparison.png'))

In [ ]:
display(result['closure'].round(4))
display(result['capture_audit'])
display(result['parameters'].round(5))

In [ ]:
for name in [
    '03_rco2_temperature_pulse_drivers.png',
    '04_liquid_ethyl_octanoate.png',
    '04_liquid_isoamyl_acetate.png',
    '05_condensate_ethyl_octanoate.png',
    '05_condensate_isoamyl_acetate.png',
    '06_parameter_stability.png',
]:
    display(Image(filename=analysis.FIGURE_DIR / name))

## Conclusiones

El ajuste del vino evalúa la estructura biológica sin contaminación del bloque de captura. El condensado evalúa por separado si las pérdidas predichas son compatibles con captura A+B completa. Un fallo de cierre no identifica eficiencia física: rechaza el contrato conjunto bajo la estructura y datos actuales.

In [ ]:
assert result['fit_validation']['calibration_domains'].eq('wine').all()
assert result['capture_audit']['fixed_total_capture_fraction'].eq(1.0).all()
assert result['capture_audit']['maximum_absolute_unrecovered_ug'].max() < 1e-8
assert result['fit_validation']['maximum_relative_mass_balance_error'].max() <= 1e-8
print('Notebook ejecutado sin errores; condensado excluido del ajuste.')